# Train, predict, score, and evaluate a Sequence CNN on PLAsTiCC

This notebook uses your `dlip_plasticc` package on top of `avocado` to:

- train a `SequenceCNNClassifier`
- predict on chunked PLAsTiCC test data
- compute the flat-weighted avocado logloss
- plot training history (`train_loss`)
- display a confusion matrix

It assumes your refactored package layout exists, including:

- `dlip_plasticc.config`
- `dlip_plasticc.features`
- `dlip_plasticc.models`
- `dlip_plasticc.pipelines.predict`
- `dlip_plasticc.pipelines.score`


In [ ]:
# Notebook parameters

CONFIG_DEFAULT_PATH = 'configs/default.toml'
CONFIG_LOCAL_PATH = 'configs/local.toml'

# Training
TRAIN_DATASET_NAME = 'plasticc_augment'
CLASSIFIER_NAME = 'my_sequence_cnn'
SEQ_LEN = 350
NUM_EPOCHS = 20
BATCH_SIZE = 64
LR = 1e-3
DROPOUT = 0.2

# Prediction
TEST_DATASET_NAME = 'plasticc_test'
TOTAL_CHUNKS = 500

# Change this list to the chunks you actually want to run.
# For a quick smoke test, use something like [0, 1, 2].
CHUNKS = [0, 1, 2]

# Output location for per-chunk and combined predictions.
OUT_DIR = 'notebook_outputs/sequence_cnn_predictions'

# Confusion matrix display options
CONFUSION_NORMALIZE = 'true'  # one of: None, 'true', 'pred', 'all'
FIGSIZE_HISTORY = (8, 5)
FIGSIZE_CONFUSION = (10, 8)


In [ ]:
# Optional path helper for notebooks when the package is not installed yet.
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / 'src' / 'dlip_plasticc').exists():
    repo_root = repo_root.parent

src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import avocado

from dlip_plasticc.config import load_config, apply_avocado_settings
from dlip_plasticc.features import PlasticcSequenceFeaturizer
from dlip_plasticc.models import SequenceCNNClassifier
from dlip_plasticc.pipelines.predict import predict_partial_from_dataset
from dlip_plasticc.pipelines.score import score_flat, align_truth_and_predictions

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report


In [ ]:
# Load config and push paths into avocado
cfg = load_config(CONFIG_DEFAULT_PATH, CONFIG_LOCAL_PATH)
apply_avocado_settings(cfg)

print('Using avocado paths:')
print('  data_directory       =', avocado.settings['data_directory'])
print('  features_directory   =', avocado.settings['features_directory'])
print('  predictions_directory=', avocado.settings['predictions_directory'])
print('  classifier_directory =', avocado.settings['classifier_directory'])


## 1. Train the Sequence CNN

This uses online sequence featurization from raw observations in the training dataset.

In [ ]:
featurizer = PlasticcSequenceFeaturizer(seq_len=SEQ_LEN)

classifier = SequenceCNNClassifier(
    name=CLASSIFIER_NAME,
    featurizer=featurizer,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    dropout=DROPOUT,
)

print(f"Loading training dataset '{TRAIN_DATASET_NAME}'...")
train_dataset = avocado.load(TRAIN_DATASET_NAME)

print('Extracting sequence raw features...')
train_dataset.extract_raw_features(featurizer)

print(f"Training classifier '{CLASSIFIER_NAME}'...")
classifier.train(
    train_dataset,
    show_progress=True,
)


In [ ]:
# Save the trained classifier
classifier.write(overwrite=True)
print('Classifier written to:', classifier.path)


## 2. Plot training history

In [ ]:
history = classifier.history.copy()
history.tail()


In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE_HISTORY)
ax.plot(history['epoch'], history['train_loss'], label='train_loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Sequence CNN training history')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()


## 3. Predict on test chunks

In [ ]:
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

combined_predictions, processed_chunks = predict_partial_from_dataset(
    classifier=classifier,
    featurizer=featurizer,
    dataset_name=TEST_DATASET_NAME,
    total_chunks=TOTAL_CHUNKS,
    chunks=CHUNKS,
    out_dir=OUT_DIR,
    metadata_only=False,
    show_progress=True,
)

print('Processed chunks:', processed_chunks)
print('Combined prediction shape:', combined_predictions.shape)
combined_predictions.head()


## 4. Score predictions with avocado flat-weighted logloss

This scores only on the overlapping labeled objects available in the metadata.

In [ ]:
flat_score, n_scored = score_flat(TEST_DATASET_NAME, combined_predictions)
print(f'Flat-weighted logloss: {flat_score:.5f}')
print(f'Objects scored:        {n_scored:,}')


## 5. Confusion matrix

In [ ]:
y_true, pred_aligned = align_truth_and_predictions(
    TEST_DATASET_NAME,
    combined_predictions,
    known_classes_only=True,
    normalize=True,
)

y_pred = pred_aligned.idxmax(axis=1)
labels = sorted(np.unique(np.concatenate([y_true.values, y_pred.values])))

cm = confusion_matrix(y_true, y_pred, labels=labels, normalize=CONFUSION_NORMALIZE)

fig, ax = plt.subplots(figsize=FIGSIZE_CONFUSION)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, xticks_rotation=45, colorbar=True, cmap='Blues', values_format='.2f')
ax.set_title('Confusion matrix on scored predictions')
plt.tight_layout()
plt.show()


In [ ]:
print(classification_report(y_true, y_pred, digits=4))


## 6. Optional: Save combined predictions to a separate CSV

Useful if you want a quick export without opening the HDF file.

In [ ]:
csv_path = Path(OUT_DIR) / 'predictions_combined.csv'
combined_predictions.to_csv(csv_path)
print('Wrote:', csv_path)
